# CSE412 Applied Project: Hospital Readmission Risk Prediction & Healthcare Analytics

**Pipeline:** UCI healthcare records → **HDFS** → **Apache Pig** → **Apache Hive** → **Apache Spark MLlib**

This notebook is designed to orchestrate a real multi-tool pipeline. Pig cleans the dataset stored in HDFS; Hive creates and queries a warehouse table and exports a modeling dataset; Spark reads that Hive-produced dataset from HDFS and trains/evaluates classifiers.

> **Environment requirement:** run Jupyter from the same **Ubuntu/WSL environment** in which Hadoop, Pig, Hive/Beeline, and Spark are configured. This is not a pure-Python simulation. External services must be running and accessible.

## Research question
Can historical encounter data reveal factors associated with 30-day readmission, and how well can Spark MLlib classifiers identify encounters readmitted within 30 days?

## Dataset
UCI **Diabetes 130-US Hospitals for Years 1999–2008**. Target definition: `<30` → 1 (readmitted within 30 days); `>30` or `NO` → 0. This is a teaching/research dataset, not a clinical decision system.

Source: https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008

## Workflow
1. Download and inspect the source CSV.
2. Copy raw data into HDFS.
3. Run Pig cleaning and write cleaned TSV records back to HDFS.
4. Create/query a Hive external table over Pig output, then export the table to an HDFS modeling directory.
5. Load the Hive-produced files in Spark; split by patient identifier to avoid encounters from the same patient crossing train/test; train Logistic Regression and Random Forest; report metrics.

**Scale note:** this dataset has about 100k encounters, which is manageable on one computer. The project's technical justification is the integrated ETL + warehouse query + distributed ML workflow, not a claim that this dataset alone is too large for a laptop. For a stricter data-volume requirement, discuss a larger permitted dataset or justified expansion with your instructor.

## 0. Before running

Start the services in your terminal using the commands appropriate to your installation. A common pseudo-distributed Hadoop setup uses `start-dfs.sh` (and, if configured, `start-yarn.sh`). Start the Hive Metastore and HiveServer2 as configured in your Hive installation. Then verify that `hdfs`, `pig`, `beeline`, and `spark-submit` are on `PATH`.

Typical checks from Ubuntu/WSL:

```bash
hdfs dfs -ls /
pig -version
beeline -u 'jdbc:hive2://localhost:10000/default' -n hive -e 'SHOW DATABASES;'
spark-submit --version
```

Commands/service names differ between installations. If a check fails, fix that service/configuration before proceeding. The notebook intentionally stops on failed pipeline stages instead of pretending a stage succeeded.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import textwrap
import urllib.request
import zipfile
import pandas as pd
from IPython.display import display, Markdown

# ---- EDIT THESE SETTINGS FOR YOUR ENVIRONMENT ----
PROJECT_DIR = Path.cwd() / "cse412_hospital_readmission"
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
SCRIPTS_DIR = PROJECT_DIR / "scripts"
LOGS_DIR = PROJECT_DIR / "logs"
for _p in (RAW_DIR, SCRIPTS_DIR, LOGS_DIR):
    _p.mkdir(parents=True, exist_ok=True)

# HDFS NameNode URI; update if your Hadoop core-site.xml uses another address/port.
HDFS_URI = "hdfs://localhost:9000"
HDFS_ROOT = "/cse412_hospital_readmission"
HDFS_RAW_DIR = f"{HDFS_URI}{HDFS_ROOT}/raw"
HDFS_PIG_OUT = f"{HDFS_URI}{HDFS_ROOT}/pig/cleaned"
HDFS_HIVE_MODEL_DIR = f"{HDFS_URI}{HDFS_ROOT}/hive/model_input"

# HiveServer2 connection; update username/URL for your installation.
HIVE_JDBC_URL = "jdbc:hive2://localhost:10000/default"
HIVE_USER = os.environ.get("HIVE_USER", "hive")
HIVE_DATABASE = "cse412_readmission"

# Set e.g. "yarn" if Spark is configured to run on YARN; local[*] is simplest for a demo.
SPARK_MASTER = "local[*]"

DATASET_ZIP_URL = "https://archive.ics.uci.edu/static/public/296/diabetes+130-us+hospitals+for+years+1999-2008.zip"
ZIP_PATH = RAW_DIR / "uci_diabetes_130.zip"
CSV_PATH = RAW_DIR / "diabetic_data.csv"

print("Project directory:", PROJECT_DIR)
print("HDFS URI:", HDFS_URI)
print("Spark master:", SPARK_MASTER)
print("Edit the settings above before proceeding if your local service addresses differ.")

In [ ]:
def run_command(args, *, check=True, cwd=None, env=None, tail=12000):
    """Run a command without hiding useful diagnostics."""
    print("$", " ".join(map(str, args)))
    result = subprocess.run(
        list(map(str, args)), cwd=cwd, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    output = result.stdout or ""
    print(output[-tail:])
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed (exit {result.returncode}): {args[0]}. See output above.")
    return result

# Check required commands exist in the Jupyter kernel's environment.
required_commands = ["hdfs", "pig", "beeline"]
command_paths = {cmd: shutil.which(cmd) for cmd in required_commands}
print("Command availability:")
for cmd, path in command_paths.items():
    print(f"  {cmd:10} -> {path or 'NOT FOUND'}")

missing = [cmd for cmd, path in command_paths.items() if path is None]
if missing:
    print("\nSTOP: these commands are not on PATH:", ", ".join(missing))
    print("Open Jupyter from your Ubuntu/WSL terminal after loading your Hadoop/Pig/Hive environment, then rerun this cell.")
else:
    run_command(["hdfs", "dfs", "-ls", "/"], check=False)
    run_command(["pig", "-version"], check=False)
    run_command(["beeline", "-u", HIVE_JDBC_URL, "-n", HIVE_USER, "-e", "SHOW DATABASES;"], check=False)

## 1. Download and inspect the dataset

The notebook downloads the official UCI ZIP archive. A short Pandas read is used only to verify the file and preview the raw schema; the cleaning, warehousing, and modeling stages are performed by the course tools.

In [ ]:
if not ZIP_PATH.exists():
    print("Downloading dataset from UCI...")
    urllib.request.urlretrieve(DATASET_ZIP_URL, ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH) as zf:
    members = zf.namelist()
    csv_member = next((m for m in members if m.lower().endswith("diabetic_data.csv")), None)
    if csv_member is None:
        raise FileNotFoundError("diabetic_data.csv was not found inside the UCI archive. Inspect ZIP_PATH and update extraction logic.")
    with zf.open(csv_member) as src, open(CSV_PATH, "wb") as dst:
        shutil.copyfileobj(src, dst)

print(f"Raw CSV: {CSV_PATH}")
print(f"Size: {CSV_PATH.stat().st_size / (1024 * 1024):.2f} MiB")
preview = pd.read_csv(CSV_PATH, nrows=5)
print("Preview shape:", preview.shape)
print("Column count:", len(preview.columns))
display(preview.head())

with open(CSV_PATH, "rb") as f:
    line_count = sum(1 for _ in f) - 1
print("Approximate encounter rows (excluding header):", f"{line_count:,}")
if len(preview.columns) != 50:
    raise ValueError(f"Expected 50 CSV columns for the supplied Pig field mapping, found {len(preview.columns)}. Check the selected UCI file/schema before continuing.")

## 2. Ingest raw CSV into HDFS

This stage copies the downloaded local source file into HDFS. Confirm that HDFS is running and `HDFS_URI` matches your NameNode configuration.

In [ ]:
run_command(["hdfs", "dfs", "-mkdir", "-p", HDFS_RAW_DIR])
run_command(["hdfs", "dfs", "-put", "-f", CSV_PATH, f"{HDFS_RAW_DIR}/diabetic_data.csv"])
run_command(["hdfs", "dfs", "-ls", HDFS_RAW_DIR])
print("INGESTION CHECKPOINT PASSED: raw CSV is in HDFS.")

## 3. Apache Pig: clean and transform the records

Pig reads from HDFS, filters invalid readmission labels and records with uncastable required numeric fields, replaces unknown race values, and produces a compact tab-separated dataset. The target `label` is 1 only when `readmitted == '<30'`.

The original file has 50 columns. This script uses their known positions from the UCI schema; if you choose another data source, re-validate the column mapping.

In [ ]:
pig_fields = ",\n    ".join(f"f{i}:chararray" for i in range(50))
pig_script = f"""\
raw = LOAD '$INPUT' USING PigStorage(',') AS (
    {pig_fields}
);

-- Remove the one CSV header row, then keep known target values only.
without_header = FILTER raw BY f0 != 'encounter_id';
known_target = FILTER without_header BY (f49 == '<30' OR f49 == '>30' OR f49 == 'NO');

-- Project the fields used by analytics/modeling, converting numeric fields to integers.
projected = FOREACH known_target GENERATE
    f1 AS patient_nbr,
    (f2 == '?' ? 'Unknown' : f2) AS race,
    f3 AS gender,
    f4 AS age,
    (int)f9 AS time_in_hospital,
    (int)f12 AS num_lab_procedures,
    (int)f14 AS num_medications,
    (int)f15 AS number_outpatient,
    (int)f16 AS number_emergency,
    (int)f17 AS number_inpatient,
    (int)f21 AS number_diagnoses,
    (f49 == '<30' ? 1 : 0) AS label;

-- Exclude invalid casts/empty identifiers before storing the cleaned TSV.
cleaned = FILTER projected BY
    patient_nbr IS NOT NULL AND patient_nbr != '' AND
    time_in_hospital IS NOT NULL AND
    num_lab_procedures IS NOT NULL AND
    num_medications IS NOT NULL AND
    number_outpatient IS NOT NULL AND
    number_emergency IS NOT NULL AND
    number_inpatient IS NOT NULL AND
    number_diagnoses IS NOT NULL;

STORE cleaned INTO '$OUTPUT' USING PigStorage('\\t');
"""
PIG_SCRIPT_PATH = SCRIPTS_DIR / "clean_hospital_data.pig"
PIG_SCRIPT_PATH.write_text(pig_script, encoding="utf-8")
print("Wrote Pig script:", PIG_SCRIPT_PATH)
print(pig_script[:1800] + "\n... (remaining script saved to file)")

# Remove the prior output directory so Pig can write on a notebook rerun.
run_command(["hdfs", "dfs", "-rm", "-r", "-f", HDFS_PIG_OUT], check=False)
run_command([
    "pig", "-x", "mapreduce",
    "-param", f"INPUT={HDFS_RAW_DIR}/diabetic_data.csv",
    "-param", f"OUTPUT={HDFS_PIG_OUT}",
    PIG_SCRIPT_PATH,
], tail=16000)
run_command(["hdfs", "dfs", "-ls", HDFS_PIG_OUT])
print("PIG CHECKPOINT PASSED: cleaned records have been written to HDFS.")

## 4. Apache Hive: create warehouse table, query it, and export model input

Hive's external table points to the actual directory written by Pig. We then query readmission rates through HiveQL and materialize the table as a tab-separated HDFS directory. Spark in the next stage reads this Hive-produced data, so the data flow is explicit: **HDFS → Pig output → Hive table/query/export → Spark**.

In [ ]:
hive_sql = f"""
CREATE DATABASE IF NOT EXISTS {HIVE_DATABASE};
DROP TABLE IF EXISTS {HIVE_DATABASE}.cleaned_encounters;
CREATE EXTERNAL TABLE {HIVE_DATABASE}.cleaned_encounters (
    patient_nbr STRING,
    race STRING,
    gender STRING,
    age STRING,
    time_in_hospital INT,
    num_lab_procedures INT,
    num_medications INT,
    number_outpatient INT,
    number_emergency INT,
    number_inpatient INT,
    number_diagnoses INT,
    label INT
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY '\\t'
STORED AS TEXTFILE
LOCATION '{HDFS_PIG_OUT}';

-- Materialize the Hive table as Spark's input; do not use a local copy of the raw CSV.
INSERT OVERWRITE DIRECTORY '{HDFS_HIVE_MODEL_DIR}'
ROW FORMAT DELIMITED
FIELDS TERMINATED BY '\\t'
SELECT patient_nbr, race, gender, age, time_in_hospital,
       num_lab_procedures, num_medications, number_outpatient,
       number_emergency, number_inpatient, number_diagnoses, label
FROM {HIVE_DATABASE}.cleaned_encounters;
"""
HIVE_SETUP_SQL = SCRIPTS_DIR / "create_hive_table_and_export.sql"
HIVE_SETUP_SQL.write_text(hive_sql, encoding="utf-8")
print("Wrote Hive SQL:", HIVE_SETUP_SQL)
run_command(["beeline", "-u", HIVE_JDBC_URL, "-n", HIVE_USER, "-f", HIVE_SETUP_SQL], tail=18000)
print("HIVE CHECKPOINT PASSED: table created and model data exported.")

# Confirm the generated Hive output directory exists in HDFS.
run_command(["hdfs", "dfs", "-ls", HDFS_HIVE_MODEL_DIR])

In [ ]:
def run_hive_query(sql):
    """Execute a HiveQL query through Beeline and display its actual output."""
    qpath = SCRIPTS_DIR / "hive_analysis_query.sql"
    qpath.write_text(sql.strip().rstrip(";") + ";\n", encoding="utf-8")
    return run_command(["beeline", "-u", HIVE_JDBC_URL, "-n", HIVE_USER, "-f", qpath], tail=12000)

print("30-day readmission rate by age group:")
run_hive_query(f"""
SELECT age, COUNT(*) AS encounter_count,
       ROUND(100.0 * AVG(CAST(label AS DOUBLE)), 2) AS readmission_rate_pct
FROM {HIVE_DATABASE}.cleaned_encounters
GROUP BY age
ORDER BY age
""")

print("Readmission rate by gender:")
run_hive_query(f"""
SELECT gender, COUNT(*) AS encounter_count,
       ROUND(100.0 * AVG(CAST(label AS DOUBLE)), 2) AS readmission_rate_pct
FROM {HIVE_DATABASE}.cleaned_encounters
GROUP BY gender
ORDER BY encounter_count DESC
""")

## 5. Apache Spark MLlib: load Hive-produced data and validate it

This stage reads the TSV exported by Hive from HDFS. The `patient_nbr` identifier is retained only for grouping the train/test split; it is excluded from model features. A deterministic hash split keeps encounters for a patient on only one side of the split.

In [ ]:
# If PySpark is missing, install a Spark version compatible with your cluster in the active Jupyter environment.
# Example only (check your Hadoop/Spark versions before running): %pip install pyspark

try:
    from pyspark.sql import SparkSession, functions as F
    from pyspark.sql.types import StructType, StructField, StringType, IntegerType
    from pyspark.ml import Pipeline
    from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
    from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
    from pyspark.ml.evaluation import BinaryClassificationEvaluator
except ImportError as exc:
    raise ImportError("PySpark is not installed in this notebook kernel. Install/configure PySpark compatible with your Spark/Hadoop setup, then rerun this cell.") from exc

spark = (SparkSession.builder
         .appName("CSE412-Hospital-Readmission")
         .master(SPARK_MASTER)
         .config("spark.hadoop.fs.defaultFS", HDFS_URI)
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

model_schema = StructType([
    StructField("patient_nbr", StringType(), True),
    StructField("race", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("age", StringType(), True),
    StructField("time_in_hospital", IntegerType(), True),
    StructField("num_lab_procedures", IntegerType(), True),
    StructField("num_medications", IntegerType(), True),
    StructField("number_outpatient", IntegerType(), True),
    StructField("number_emergency", IntegerType(), True),
    StructField("number_inpatient", IntegerType(), True),
    StructField("number_diagnoses", IntegerType(), True),
    StructField("label", IntegerType(), True),
])

model_data = (spark.read
              .option("sep", "\t")
              .option("header", "false")
              .schema(model_schema)
              .csv(HDFS_HIVE_MODEL_DIR))

print("Rows read by Spark:", f"{model_data.count():,}")
model_data.printSchema()
display(model_data.limit(8).toPandas())
print("Target distribution:")
display(model_data.groupBy("label").count().orderBy("label").toPandas())

In [ ]:
# Deterministic patient-level split: same patient_nbr always maps to the same partition.
partition_key = F.pmod(F.xxhash64(F.col("patient_nbr")), F.lit(100))
train_raw = model_data.filter(partition_key < 80).cache()
test_raw = model_data.filter(partition_key >= 80).cache()

train_count, test_count = train_raw.count(), test_raw.count()
print(f"Train rows: {train_count:,} ({train_count/(train_count+test_count):.1%})")
print(f"Test rows:  {test_count:,} ({test_count/(train_count+test_count):.1%})")
if train_count == 0 or test_count == 0:
    raise RuntimeError("The patient-level split produced an empty partition. Check input and split settings.")

categorical_cols = ["race", "gender", "age"]
numeric_cols = [
    "time_in_hospital", "num_lab_procedures", "num_medications",
    "number_outpatient", "number_emergency", "number_inpatient",
    "number_diagnoses",
]
indexed_cols = [c + "_idx" for c in categorical_cols]
encoded_cols = [c + "_ohe" for c in categorical_cols]

preprocessor = Pipeline(stages=[
    StringIndexer(inputCols=categorical_cols, outputCols=indexed_cols, handleInvalid="keep"),
    OneHotEncoder(inputCols=indexed_cols, outputCols=encoded_cols, handleInvalid="keep"),
    VectorAssembler(inputCols=numeric_cols + encoded_cols, outputCol="features", handleInvalid="keep"),
])
preprocessor_model = preprocessor.fit(train_raw)
train_features = preprocessor_model.transform(train_raw).select("patient_nbr", "label", "features").cache()
test_features = preprocessor_model.transform(test_raw).select("patient_nbr", "label", "features").cache()
print("Preprocessing completed. Numeric fields + one-hot encoded categorical fields are in `features`.")

## 6. Train two Spark MLlib models

We compare Logistic Regression (baseline) with Random Forest. Accuracy alone can be misleading because 30-day readmission is the minority class, so the evaluation includes precision, recall, F1, and ROC-AUC.

In [ ]:
def calculate_metrics(predictions, model_name):
    """Compute positive-class precision/recall/F1 plus accuracy and ROC-AUC."""
    pred = predictions.select(
        F.col("label").cast("int").alias("label"),
        F.col("prediction").cast("int").alias("prediction")
    ).cache()
    tp = pred.filter((F.col("label") == 1) & (F.col("prediction") == 1)).count()
    fp = pred.filter((F.col("label") == 0) & (F.col("prediction") == 1)).count()
    fn = pred.filter((F.col("label") == 1) & (F.col("prediction") == 0)).count()
    tn = pred.filter((F.col("label") == 0) & (F.col("prediction") == 0)).count()
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if precision + recall else 0.0
    total = tp + fp + fn + tn
    accuracy = (tp + tn) / total if total else 0.0
    auc = BinaryClassificationEvaluator(
        labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
    ).evaluate(predictions)
    pred.unpersist()
    return {
        "model": model_name,
        "accuracy": accuracy,
        "precision_readmit": precision,
        "recall_readmit": recall,
        "f1_readmit": f1,
        "roc_auc": auc,
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
    }

lr = LogisticRegression(
    featuresCol="features", labelCol="label", maxIter=30, regParam=0.05,
    elasticNetParam=0.0, threshold=0.5
)
print("Training Logistic Regression...")
lr_model = lr.fit(train_features)
lr_predictions = lr_model.transform(test_features).cache()

rf = RandomForestClassifier(
    featuresCol="features", labelCol="label", numTrees=50,
    maxDepth=8, seed=42, featureSubsetStrategy="sqrt"
)
print("Training Random Forest (50 trees)...")
rf_model = rf.fit(train_features)
rf_predictions = rf_model.transform(test_features).cache()

metrics = [calculate_metrics(lr_predictions, "Logistic Regression"),
           calculate_metrics(rf_predictions, "Random Forest")]
metrics_df = pd.DataFrame(metrics).set_index("model")
display(metrics_df.round(4))

print("Interpretation reminder: compare positive-class recall and precision as well as ROC-AUC; select a threshold based on the task rather than assuming 0.5 is clinically optimal.")

In [ ]:
import matplotlib.pyplot as plt

plot_metrics = metrics_df[["precision_readmit", "recall_readmit", "f1_readmit", "roc_auc"]]
ax = plot_metrics.plot(kind="bar", figsize=(10, 5))
ax.set_title("Spark MLlib model comparison")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

# Example predictions only; patient identifier is shown for traceability in the demo, not used as a feature.
display(rf_predictions.select("patient_nbr", "label", "prediction", "probability")
        .limit(12).toPandas())

## 7. Save results and verify the end-to-end pipeline

Results include model metrics and sample predictions. These are local output artifacts for the report/demo; the modeling data itself originated from Hive's HDFS export.

In [ ]:
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
metrics_path = RESULTS_DIR / "model_metrics.csv"
metrics_df.reset_index().to_csv(metrics_path, index=False)

prediction_path = RESULTS_DIR / "sample_predictions.csv"
(rf_predictions.select("patient_nbr", "label", "prediction", "probability")
 .limit(1000).toPandas().to_csv(prediction_path, index=False))

# Record the main tool settings for reproducibility.
run_info = {
    "dataset": "UCI Diabetes 130-US Hospitals for Years 1999-2008",
    "hdfs_uri": HDFS_URI,
    "hdfs_root": HDFS_ROOT,
    "pig_output": HDFS_PIG_OUT,
    "hive_model_export": HDFS_HIVE_MODEL_DIR,
    "hive_jdbc_url": HIVE_JDBC_URL,
    "spark_version": spark.version,
    "spark_master": SPARK_MASTER,
    "raw_csv_rows_approx": line_count,
    "spark_model_rows": model_data.count(),
    "train_rows": train_count,
    "test_rows": test_count,
}
import json
(RESULTS_DIR / "run_info.json").write_text(json.dumps(run_info, indent=2), encoding="utf-8")

print("Saved:", metrics_path)
print("Saved:", prediction_path)
print("Saved:", RESULTS_DIR / "run_info.json")
print("\nEND-TO-END PIPELINE COMPLETE: HDFS ingestion → Pig cleaning → Hive table/query/export → Spark MLlib.")

## 8. Troubleshooting checklist

- **`hdfs` connection refused / SafeMode:** start NameNode/DataNode and verify `hdfs dfs -ls /`. Keep `HDFS_URI` consistent with `fs.defaultFS` in `core-site.xml`.
- **Pig cannot run MapReduce:** check `HADOOP_HOME`, `HADOOP_CONF_DIR`, `JAVA_HOME`, and whether Hadoop services are healthy. Make sure the `pig` executable is from the intended installation.
- **Beeline connection refused:** start the Hive Metastore and HiveServer2 and verify `HIVE_JDBC_URL` and `HIVE_USER`. Authentication/config can differ from the example.
- **Hive table has zero rows:** inspect `hdfs dfs -ls` and a small sample from `HDFS_PIG_OUT`; check Pig's field positions and target values.
- **Hive cannot find the output:** ensure Pig and Hive refer to the same NameNode filesystem/URI and that the Pig output folder exists.
- **Spark cannot read HDFS:** run the notebook from an environment with Hadoop client configs and compatible Spark/PySpark. Check `HDFS_URI` and Hadoop JAR/config compatibility.
- **CSV parse or field-count issue:** the Pig script assumes the UCI `diabetic_data.csv` schema with 50 comma-delimited fields and no embedded CSV delimiters. Validate columns before using a different dataset.
- **Model performance looks modest:** inspect class balance, compare positive-class recall/precision and ROC-AUC, test class weighting or threshold tuning, and avoid presenting predictive performance as causal evidence or clinical guidance.

## Suggested 10-minute demo

1. Show raw dataset and HDFS ingestion.
2. Run Pig script and inspect cleaned output directory.
3. Show Hive table creation plus age/gender readmission queries.
4. Show Spark row count, patient-level split, model training, metrics and model comparison plot.
5. Explain one limitation and the role of each tool. Keep your report honest about what actually ran and which versions were used.